# Python Abstract Classes and Dataclasses — Interview Preparation

Covers: ABC, abstractmethod, @dataclass, field(), __post_init__, frozen dataclasses, slots, and interview Q&A.

## 1. Abstract Base Classes

In [ ]:
from abc import ABC, abstractmethod
import math

class Shape(ABC):
    """Abstract base class — cannot be instantiated directly"""

    @abstractmethod
    def area(self) -> float:
        """Must be implemented by all subclasses"""
        pass

    @abstractmethod
    def perimeter(self) -> float:
        pass

    # Concrete method — shared by all subclasses
    def describe(self) -> str:
        return f'{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}'


# Cannot instantiate abstract class
try:
    s = Shape()
except TypeError as e:
    print('Error:', e)


class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius

    def area(self) -> float:
        return math.pi * self.radius ** 2

    def perimeter(self) -> float:
        return 2 * math.pi * self.radius


class Rectangle(Shape):
    def __init__(self, width: float, height: float):
        self.width = width
        self.height = height

    def area(self) -> float:
        return self.width * self.height

    def perimeter(self) -> float:
        return 2 * (self.width + self.height)


shapes = [Circle(5), Rectangle(4, 6), Circle(3)]
for s in shapes:
    print(s.describe())

print('Total area:', sum(s.area() for s in shapes))

## 2. @dataclass — Basic Usage

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Point:
    x: float
    y: float

# Auto-generated: __init__, __repr__, __eq__
p1 = Point(1.0, 2.0)
p2 = Point(1.0, 2.0)
p3 = Point(3.0, 4.0)

print('p1:', p1)              # __repr__
print('p1 == p2:', p1 == p2)  # __eq__ — True
print('p1 == p3:', p1 == p3)  # False

# Without @dataclass, you'd need to write all this manually!
class PointManual:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __repr__(self):
        return f'PointManual(x={self.x!r}, y={self.y!r})'
    def __eq__(self, other):
        return isinstance(other, PointManual) and self.x == other.x and self.y == other.y

## 3. field() and Mutable Defaults

In [ ]:
from dataclasses import dataclass, field
from typing import List

# WRONG — mutable default raises ValueError
try:
    @dataclass
    class Bad:
        items: list = []  # ValueError!
except TypeError as e:
    print('Error:', e)

# RIGHT — use field(default_factory=...)
@dataclass
class Student:
    name: str
    age: int
    grades: List[float] = field(default_factory=list)
    # repr=False: exclude from __repr__
    # compare=False: exclude from __eq__
    notes: str = field(default='', repr=False, compare=False)

s1 = Student('Alice', 20)
s2 = Student('Bob', 22)
s1.grades.append(95.0)
s2.grades.append(88.0)

print('s1:', s1)  # notes not shown (repr=False)
print('s2:', s2)
print('s1.grades:', s1.grades)  # independent lists!

## 4. __post_init__ and InitVar

In [ ]:
from dataclasses import dataclass, field, InitVar

@dataclass
class Rectangle:
    width: float
    height: float
    area: float = field(init=False)  # not in __init__, computed in __post_init__

    def __post_init__(self):
        """Called after __init__ — for validation and derived fields"""
        if self.width <= 0 or self.height <= 0:
            raise ValueError('Dimensions must be positive')
        self.area = self.width * self.height


r = Rectangle(4, 5)
print('Rectangle:', r)
print('Area:', r.area)  # computed in __post_init__

try:
    bad = Rectangle(-1, 5)
except ValueError as e:
    print('Error:', e)


# InitVar — passed to __post_init__ but not stored as field
@dataclass
class User:
    name: str
    raw_password: InitVar[str]  # not stored
    password_hash: str = field(init=False)

    def __post_init__(self, raw_password: str):
        import hashlib
        self.password_hash = hashlib.sha256(raw_password.encode()).hexdigest()[:16]


u = User('Alice', 'secret123')
print('User:', u)
# raw_password is NOT stored — only the hash

## 5. Frozen Dataclasses

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Color:
    r: int
    g: int
    b: int

    def __post_init__(self):
        for name, val in [('r', self.r), ('g', self.g), ('b', self.b)]:
            if not 0 <= val <= 255:
                raise ValueError(f'{name} must be 0-255, got {val}')

    def to_hex(self):
        return f'#{self.r:02x}{self.g:02x}{self.b:02x}'


RED = Color(255, 0, 0)
GREEN = Color(0, 255, 0)
BLUE = Color(0, 0, 255)

print('RED:', RED)
print('RED hex:', RED.to_hex())

# Frozen — cannot modify
try:
    RED.r = 100
except Exception as e:
    print('Error:', type(e).__name__, e)

# Frozen dataclasses are hashable!
print('hash(RED):', hash(RED))
color_set = {RED, GREEN, BLUE, RED}  # duplicate removed
print('color_set:', color_set)
color_map = {RED: 'red', GREEN: 'green'}
print('color_map[RED]:', color_map[RED])

## 6. Dataclass with order=True

In [ ]:
from dataclasses import dataclass

@dataclass(order=True)
class Version:
    major: int
    minor: int
    patch: int

    def __str__(self):
        return f'{self.major}.{self.minor}.{self.patch}'


v1 = Version(1, 2, 3)
v2 = Version(2, 0, 0)
v3 = Version(1, 3, 0)

print('v1 < v2:', v1 < v2)   # True
print('v1 < v3:', v1 < v3)   # True (1.2.3 < 1.3.0)

versions = [v2, v1, v3, Version(1, 2, 4)]
print('Sorted:', [str(v) for v in sorted(versions)])